In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import pandas as pd

In [5]:
# -------------------------
# OpenAI 설정
# -------------------------

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [ ]:
# -------------------------
# 음성 파일 transcription + 화자 분리
# -------------------------

audio_file_path = "audio/싼기타_비싼기타.mp3"

with open(audio_file_path, "rb") as audio_file:
    result = client.audio.transcriptions.create(
        model="gpt-4o-transcribe-diarize",
        file=audio_file,
        chunking_strategy="auto",
        response_format="diarized_json",
    )


In [15]:
# -------------------------
# diarization 결과 확인
# -------------------------

start_end_text = []

for segment in result.segments:

    start_end_text.append([
        segment.start,
        segment.end,
        segment.speaker,
        segment.text,
    ])


In [8]:
# -------------------------
# DataFrame 생성
# -------------------------

df_diarized = pd.DataFrame(
    start_end_text,
    columns=["start", "end", "speaker_id", "text"]
)

display(df_diarized)

,start,end,speaker_id,text
0,0.724,25.724,A,지금부터 저랑 역할극을 합시다 역할극을 스탠딩 코미디 스타일로 할 건데 토론을 하...
1,25.972,27.722,A,저랑 토론해보면 좋을 것 같아요
2,28.222,30.122,A,둘 중에 어떤 역할 맡으실래요?
3,32.072,32.872,B,좋습니다.
4,33.272,36.822,B,그럼 제가 쌍기타로 시작하는 게 좋다는 입장을 맡아볼게요.
...,...,...,...,...
80,414.544,417.194,B,계속해서 이야기 나누고 싶으시면 편하게 말씀해 주세요.
81,417.344,420.944,A,아니요 화나셨는데 굳이 더 할 필요 없죠 그만 이쯤 하시죠
82,423.344,424.094,B,알겠습니다.
83,424.444,428.044,B,언제든 다시 이야기 나누고 싶으실 때 편하게 말씀해 주세요.


In [9]:
# -------------------------
# 화자 변경 기준 number 생성
# -------------------------

df_diarized["number"] = 0

for i in range(1, len(df_diarized)):

    if (
        df_diarized.at[i, "speaker_id"]
        != df_diarized.at[i - 1, "speaker_id"]
    ):
        df_diarized.at[i, "number"] = (
            df_diarized.at[i - 1, "number"] + 1
        )

    else:
        df_diarized.at[i, "number"] = (
            df_diarized.at[i - 1, "number"]
        )


In [10]:
# -------------------------
# 화자별 연속 구간 그룹화
# -------------------------

df_grouped = df_diarized.groupby("number").agg(
    start=("start", "min"),
    end=("end", "max"),
    speaker_id=("speaker_id", "first"),
    text=("text", lambda x: " ".join(x))
)

In [11]:
# -------------------------
# duration 계산
# -------------------------

df_grouped["duration"] = (
    df_grouped["end"] - df_grouped["start"]
)

In [12]:
# -------------------------
# index 초기화
# -------------------------

df_grouped = df_grouped.reset_index(drop=True)

In [13]:
# -------------------------
# 결과 확인
# -------------------------

display(df_grouped)


,start,end,speaker_id,text,duration
0,0.724,30.122,A,지금부터 저랑 역할극을 합시다 역할극을 스탠딩 코미디 스타일로 할 건데 토론을 하...,29.398
1,32.072,41.122,B,좋습니다. 그럼 제가 쌍기타로 시작하는 게 좋다는 입장을 맡아볼게요. 그럼 성...,9.050
2,41.472,41.922,A,네 맞아요.,0.450
3,41.922,42.372,B,준비되셨나요?,0.450
4,42.372,43.772,A,네 됐어요. 시작하시죠.,1.400
5,45.672,66.900,B,좋아요. 먼저 쌍기타로 시작하는 게 좋은 이유를 말씀드리겠습니다. 초보자일 때...,21.228
6,66.950,82.600,A,아 저는 지금 말에 어폐가 있다고 생각해요. 왜냐하면 어차피 지금 비싼 기타로 ...,15.650
7,84.150,102.272,B,그런데 비싼 기타로 스타트하면 혹시라도 흠집이 나거나 실수할 때 부담이 더 크지 ...,18.122
8,103.172,117.272,A,아니 근데 어차피 그 비싼 기타를 살 건데 싼 기타를 뭐하러 더 삽니까 그리고 기...,14.100
9,119.540,138.390,B,하하 기타를 망치로 치진 않지만 그래도 초보자들은 실수도 많고 조심스럽게 다루기 ...,18.850


In [16]:
# -------------------------
# CSV 저장
# -------------------------

df_grouped.to_csv(
    "audio/싼기타_비싼기타_diarized_by_openai.csv",
    index=False,
    sep=","
)

> 결과를 보니 자동으로 화자분리가 된다

> gpt-4o-transcribe-diarize는 화자 분리(diarization)가 내장된 모델이라 별도로 pyannote를 실행할 필요가 없다